# TürkResearcher — Stage 1 Day 3: Rebuild Chroma index with `trakad-embed-v2`

Re-encodes all **633K Turkish theses** with our new fine-tuned embedder and writes a fresh Chroma vector store. This is the artifact the live demo + 30-question eval will hit.

**Önce:** Runtime → Change runtime type → **A100 GPU** seç (T4 de çalışır ama ~3x yavaş).

**Akış (~25-35 dk):**
1. Setup + GPU
2. Workspace + HF auth
3. Pull `abstracts_filtered.parquet` + `trakad-embed-v2` from HF Hub
4. Encode 633K docs (title + abstract) → 768-dim float32 vectors (~10-15 dk)
5. Build Chroma index with `hnsw:space=cosine` + precomputed embeddings (~5 dk)
6. Smoke retrieval (5 sample queries — kalitatif kontrol)
7. Tar + push to `hakansabunis/tr-academic-research-agent-index` under `chroma_db_v2/`

## 1. Setup

In [ ]:
!nvidia-smi
!pip install -q sentence-transformers>=3.0 chromadb>=0.5 huggingface_hub pyarrow pandas tqdm

## 2. Workspace + HF auth

In [ ]:
import os
WORK_DIR = '/content/index_build'
os.makedirs(WORK_DIR, exist_ok=True)
print('Working dir:', WORK_DIR)
!df -h /content | tail -1

In [ ]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass('HF write token: ')
login(hf_token, add_to_git_credential=False)

## 3. Pull parquet + trakad-embed-v2

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

PARQUET_PATH = hf_hub_download(
    repo_id='hakansabunis/tr-academic-research-agent-index',
    filename='abstracts_filtered.parquet',
    repo_type='dataset',
    local_dir=WORK_DIR,
)
print('Parquet:', PARQUET_PATH, f'({os.path.getsize(PARQUET_PATH)/1024/1024:.0f} MB)')

MODEL_DIR = snapshot_download(
    repo_id='hakansabunis/trakad-embed-v2',
    repo_type='model',
    local_dir=f'{WORK_DIR}/trakad-embed-v2',
)
print('Model dir:', MODEL_DIR)

## 4. Encode 633K docs (title + abstract)

Document format = same as v1 index for fair comparison: `title_tr + "\n\n" + abstract_tr`. Embeddings are L2-normalized (cosine on Chroma reduces to dot product, faster).

**Memory:** 633K × 768 × 4 bytes ≈ 1.9 GB → fits in A100 RAM. Encoded as a single in-memory numpy array, then written to disk for safety.

**Hız (A100 + batch=256):** ~1000 docs/sec → ~10-12 dk.

In [ ]:
import pandas as pd
import numpy as np
import torch, gc
from sentence_transformers import SentenceTransformer

df = pd.read_parquet(PARQUET_PATH)
print(f'Loaded {len(df):,} rows')

df['title_tr']    = df['title_tr'].fillna('').astype(str).str.strip()
df['abstract_tr'] = df['abstract_tr'].fillna('').astype(str).str.strip()

# Same doc format as v1 index for apples-to-apples eval
def make_doc(t, a):
    if t:
        return f'{t}\n\n{a}'
    return a

docs = [make_doc(t, a) for t, a in zip(df['title_tr'].values, df['abstract_tr'].values)]
ids  = df['tez_no'].astype(str).tolist()
print(f'Prepared {len(docs):,} (id, doc) pairs')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading trakad-embed-v2 on {device}...')
model = SentenceTransformer(MODEL_DIR, device=device)
model.max_seq_length = 256

import time; t0 = time.time()
embeddings = model.encode(
    docs,
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print(f'Encoded in {(time.time()-t0)/60:.1f} min · shape={embeddings.shape} · dtype={embeddings.dtype}')

# Persist embeddings to disk in case Chroma indexing fails — re-runnable without re-encode
EMB_PATH = f'{WORK_DIR}/embeddings_v2.npy'
np.save(EMB_PATH, embeddings)
print(f'Saved embeddings -> {EMB_PATH} ({os.path.getsize(EMB_PATH)/1024/1024:.0f} MB)')

# Free model GPU memory before Chroma indexing
del model; gc.collect(); torch.cuda.empty_cache()

## 5. Build Chroma index from precomputed embeddings

`hnsw:space=cosine` to match v1 layout. We pass `embeddings=` directly so Chroma doesn't try to re-encode (which would be ~10x slower).

In [ ]:
import chromadb
from tqdm import tqdm

PERSIST_DIR = f'{WORK_DIR}/chroma_db_v2'
COLLECTION  = 'turkish_theses'

# Same metadata schema as v1 build_index.py
def row_meta(row):
    def s(k):
        v = row.get(k)
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return ''
        return str(v)
    try:
        year_int = int(row.get('year')) if row.get('year') is not None and not pd.isna(row.get('year')) else 0
    except (TypeError, ValueError):
        year_int = 0
    try:
        pages_int = int(row.get('pages')) if row.get('pages') is not None and not pd.isna(row.get('pages')) else 0
    except (TypeError, ValueError):
        pages_int = 0
    return {
        'tez_no': s('tez_no'), 'title_tr': s('title_tr'), 'title_en': s('title_en'),
        'author': s('author'), 'advisor': s('advisor'), 'location': s('location'),
        'subject': s('subject'), 'year': year_int, 'pages': pages_int,
        'degree': s('degree'), 'pdf_url': s('pdf_url'), 'language': s('language'),
    }

client = chromadb.PersistentClient(path=PERSIST_DIR)
# Wipe any partial collection from a previous attempt
try:
    client.delete_collection(COLLECTION)
    print('Removed prior collection')
except Exception:
    pass

coll = client.create_collection(
    name=COLLECTION,
    metadata={'hnsw:space': 'cosine'},
    # No embedding_function -> we pass embeddings explicitly
)

BATCH = 5000
t0 = time.time()
for i in tqdm(range(0, len(docs), BATCH), desc='chroma add'):
    j = min(i + BATCH, len(docs))
    batch_ids   = ids[i:j]
    batch_docs  = docs[i:j]
    batch_metas = [row_meta(df.iloc[k]) for k in range(i, j)]
    batch_embs  = embeddings[i:j].tolist()
    coll.add(
        ids=batch_ids,
        documents=batch_docs,
        metadatas=batch_metas,
        embeddings=batch_embs,
    )

print(f'Indexed {coll.count():,} records in {(time.time()-t0)/60:.1f} min')
print(f'Persist dir: {PERSIST_DIR}')
!du -sh $PERSIST_DIR

## 6. Smoke retrieval — query the new index

In [ ]:
# Encode queries manually (Chroma 0.5+ refuses to attach an ef to a collection
# created without one — the persisted config says 'default'). We bypass by
# passing query_embeddings= directly.
from sentence_transformers import SentenceTransformer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
qmodel = SentenceTransformer(MODEL_DIR, device=device)
qmodel.max_seq_length = 256

client2 = chromadb.PersistentClient(path=PERSIST_DIR)
coll2 = client2.get_collection(name=COLLECTION)
print(f'Reopened collection · count={coll2.count():,}')

queries = [
    'derin öğrenme ile sel tahmini',
    'Türkçe doğal dil işleme metodları',
    'kalp damar hastalıkları teşhis yöntemleri',
    'yenilenebilir enerji rüzgar türbin',
    'üniversite öğrencilerinin akademik başarısı',
]

for q in queries:
    qv = qmodel.encode(q, normalize_embeddings=True).tolist()
    res = coll2.query(query_embeddings=[qv], n_results=3)
    print(f'\nQ: {q!r}')
    for doc, meta, dist in zip(res['documents'][0], res['metadatas'][0], res['distances'][0]):
        title = meta.get('title_tr', '')[:90]
        print(f'  [d={dist:.3f}] {title}')

## 7. Push to HF Hub

Uploads `chroma_db_v2/` next to the existing `chroma_db/` in the same dataset repo. v1 stays untouched so we can run side-by-side comparisons in Day 4 eval.

In [ ]:
from huggingface_hub import HfApi

REPO = 'hakansabunis/tr-academic-research-agent-index'
api = HfApi(token=hf_token)

print(f'Uploading {PERSIST_DIR} -> {REPO}/chroma_db_v2/ ...')
api.upload_folder(
    folder_path=PERSIST_DIR,
    path_in_repo='chroma_db_v2',
    repo_id=REPO,
    repo_type='dataset',
    commit_message='Add chroma_db_v2/ — index built with trakad-embed-v2 (633K theses, fine-tuned mpnet)',
    token=hf_token,
)
print(f'\n✓ Done. https://huggingface.co/datasets/{REPO}/tree/main/chroma_db_v2')